# ⚡ Just-in-time compilation with numba

*Section 14.4 of the notes.*

Python is slow at loops because the interpreter handles one instruction at a
time: for every step it looks up the types of the values, finds the right
operation, and creates a Python object for the result (Chapter 13). NumPy
avoids this by running whole-array operations in compiled C code. But some
loops cannot be written as array operations, for example a **time loop**, where
every step needs the result of the previous one. 🔁

**numba** solves this by **compiling** a Python function to machine code, the
instructions that the processor runs directly. 🛠️ You add one line above the
function, `@njit`, and numba does the rest:

* it compiles the function **the first time it is called**, hence *just in
  time*;
* it compiles a version for the **types of the arguments** of that call, for
  example "one-dimensional arrays of 64-bit floats";
* later calls with the same types run the machine code directly, without the
  interpreter.

This notebook measures the cost of compiling, the speed of the result, and
shows what numba cannot compile.

## ⚙️ Setup

Timings use `%timeit -o -q` and `.best`, as in the earlier notebooks. Only in
Section 1 we time single calls with `time.perf_counter()`, because there we
want to see the first call, which `%timeit` would hide among the repeats.

In [ ]:
import math
import time

import numpy as np
import scipy.special
from numba import njit, prange
from numba.core.errors import TypingError

rng = np.random.default_rng(0)

## 1. 🐣 The first call compiles

Our example is the periodic second difference of Chapter 2, written as a
plain loop over the grid points. The only change to ordinary Python is the
line `@njit` above the function.

We call it twice with the same arrays and time each call separately.

🤔 *Guess: how much slower is the first call?*

In [ ]:
@njit
def stencil_numba(u, out):
    d = u.size
    for j in range(d):
        out[j] = u[(j + 1) % d] - 2 * u[j] + u[j - 1]
    return out


d = 1_000_000
u = rng.standard_normal(d)
out = np.empty_like(u)

t0 = time.perf_counter()
stencil_numba(u, out)
t_first = time.perf_counter() - t0

t0 = time.perf_counter()
stencil_numba(u, out)
t_second = time.perf_counter() - t0

print(f"first call  {t_first * 1e3:8.1f} ms   (compiles, then runs)")
print(f"second call {t_second * 1e3:8.1f} ms   (runs the machine code)")
print("compiled for:", stencil_numba.signatures)

💡 **What happened?** The first call takes hundreds of times longer than the
second: almost all of it is compiling. The second call runs the machine
code directly.

`signatures` lists the argument types the function has been compiled for:
here two one-dimensional (`1`) arrays of `float64` stored in C order (`'C'`,
Chapter 13).

What if we call the function with arrays of another type, `float32`? numba
cannot reuse the machine code made for `float64`, so it compiles a **second
version**, and the first call with the new types is slow again.

In [ ]:
stencil_numba(u.astype(np.float32), out.astype(np.float32))
print("compiled for:", stencil_numba.signatures)

💡 Now there are two signatures, one per type. 👉 Remember this when you
time numba code: always call it once first, with the same types, so that the
compilation is not counted (the first-call rule of `04-measuring`).

## 2. 🏁 A race of four versions

The same second difference, written in four ways:

1. 🐍 **Python loop**: the loop above without `@njit`;
2. 🧮 **NumPy slices**: three shifted views combined in one array expression
   (Chapter 13);
3. ⚡ **numba**: the compiled loop of Section 1;
4. ⚡⚡ **numba parallel**: the same loop with `@njit(parallel=True)` and
   `prange` instead of `range`. `prange` tells numba that the steps of the loop
   are independent, so it may split them among threads, one group of grid
   points per core. Compiled code does not need the interpreter lock, so these
   threads really run at the same time.

First we check that all versions give the same result, then we time them per
grid point.

🤔 *Guess the order, from slowest to fastest!*

In [ ]:
def stencil_loop(u, out):
    d = u.size
    for j in range(d):
        out[j] = u[(j + 1) % d] - 2 * u[j] + u[j - 1]
    return out


def stencil_slices(u, out):
    out[1:-1] = u[2:] - 2 * u[1:-1] + u[:-2]
    out[0] = u[1] - 2 * u[0] + u[-1]
    out[-1] = u[0] - 2 * u[-1] + u[-2]
    return out


@njit(parallel=True)
def stencil_parallel(u, out):
    d = u.size
    for j in prange(d):  # independent steps, shared among threads
        out[j] = u[(j + 1) % d] - 2 * u[j] + u[j - 1]
    return out


reference = stencil_slices(u, np.empty_like(u))
print(
    "all versions agree:",
    np.allclose(reference, stencil_loop(u, np.empty_like(u)))
    and np.allclose(reference, stencil_numba(u, np.empty_like(u)))
    and np.allclose(
        reference, stencil_parallel(u, np.empty_like(u))
    ),  # also compiles it
)

for name, fn in [
    ("Python loop", stencil_loop),
    ("NumPy slices", stencil_slices),
    ("numba", stencil_numba),
    ("numba parallel", stencil_parallel),
]:
    t = %timeit -o -q fn(u, out)
    print(f"{name:16s} {1e9 * t.best / d:8.2f} ns per grid point")

💡 **What happened?**

* The **Python loop** is hundreds of times slower than everything else: one
  trip through the interpreter per grid point. 🐌
* **numba** beats the **NumPy slices**, even though both run compiled code.
  The NumPy expression is computed one operation at a time, and every
  operation writes a temporary array that the next one reads back
  (Chapter 13). The compiled loop goes over the grid **once**, keeps the
  intermediate values in the processor, and writes only the result.
* **numba parallel** gains less than the number of cores. Each grid point needs
  only a few additions, so the cores spend most of their time waiting for
  numbers to arrive from memory, and one call is so short (a fraction of a
  millisecond) that handing the work to the threads is a noticeable part of
  it.

👉 numba pays off most for loops that NumPy cannot express in one pass, and
`parallel=True` pays off most when there is a lot of arithmetic per number.

## 3. 🔁 A time loop

Now the case numba is made for. One forward Euler step of the Allen–Cahn
equation on the periodic grid of Chapter 2 is

$$ u_j \leftarrow u_j + \Delta t \,\bigl( D\,(u_{j+1} - 2u_j + u_{j-1})/\Delta x^2 + u_j - u_j^3 \bigr), $$

and we take 2000 such steps. The loop over time **cannot be vectorized**:
step $k+1$ needs the result of step $k$.

* 🧮 **NumPy**: each step is an array expression. Python runs the time loop,
  and every step creates several temporary arrays (two `np.roll`s, the
  products, the sums).
* ⚡ **numba**: the **whole** time loop, including the loop over grid points,
  is compiled. Two arrays `v` and `w` take turns: each step reads `v` and
  writes `w`, and then the line `v, w = w, v` swaps the two names, so no array
  is ever allocated inside the loop.

`@njit(cache=True)` also saves the machine code in a file on disk, so that
the next time you open the notebook the function does not need to be compiled
again. 💾 For a function in a `.py` file the file goes into the folder
`__pycache__` next to it; for a function defined in a notebook, into the
folder `numba_cache` inside IPython's own cache folder.

🤔 *Guess the ratio between the two!*

In [ ]:
def step_numpy(u, dt, dx, diffusion):
    lap = (np.roll(u, -1) - 2 * u + np.roll(u, 1)) / dx**2
    return u + dt * (diffusion * lap + u - u**3)


def march_numpy(u, dt, dx, diffusion, steps):
    for _ in range(steps):
        u = step_numpy(u, dt, dx, diffusion)
    return u


@njit(cache=True)
def march_numba(u, dt, dx, diffusion, steps):
    d = u.size
    v = u.copy()  # the current state
    w = np.empty_like(u)  # the next state
    for _ in range(steps):
        for j in range(d):
            lap = (v[(j + 1) % d] - 2 * v[j] + v[(j - 1) % d]) / dx**2
            w[j] = v[j] + dt * (diffusion * lap + v[j] - v[j] ** 3)
        v, w = w, v  # the new state becomes the current one
    return v


d = 2048
x = -1 + 2 * np.arange(d) / d
u0 = x**2 * np.cos(np.pi * x)
dx, dt, steps = 2 / d, 0.005, 2_000
diffusion = 2e-5  # the diffusion coefficient D

a = march_numpy(u0, dt, dx, diffusion, steps)
b = march_numba(u0, dt, dx, diffusion, steps)  # the first call compiles
print("same result:", np.allclose(a, b))

t_np = %timeit -o -q march_numpy(u0, dt, dx, diffusion, steps)
t_nb = %timeit -o -q march_numba(u0, dt, dx, diffusion, steps)
print(
    f"numpy {t_np.best * 1e3:7.1f} ms\n"
    f"numba {t_nb.best * 1e3:7.1f} ms\n"
    f"ratio {t_np.best / t_nb.best:.1f}x"
)

💡 **What happened?** The compiled time loop is several times faster. The
grid has only 2048 points, so each NumPy step is short, and the fixed costs of
every step (returning to Python, allocating and freeing the temporaries) add up
over 2000 steps. numba pays none of them.

Neither version changes `u0`: `march_numpy` builds new arrays in every step,
and `march_numba` works on a copy. This is why the same `u0` can be passed to
every call.

## 4. 🚫 What numba refuses

numba compiles only the parts of Python and NumPy that it knows: numbers,
arrays, loops, most of NumPy's functions, and the `math` module. It does
**not** know other libraries such as SciPy or pandas, or your own classes.

When it meets something it cannot compile, it stops with a `TypingError`
that names the line. It never runs the function slowly in Python instead
without telling you. 👍

Here a function calls `scipy.special.erf`, the error function from SciPy.

In [ ]:
@njit
def uses_scipy(x):
    return scipy.special.erf(x)


try:
    uses_scipy(0.5)
except TypingError as err:
    first_lines = str(err).strip().splitlines()[:3]
    print("compilation failed:")
    print("\n".join(first_lines))

💡 The message says that numba does not know the attribute `erf` of the
module `scipy.special`. The fix is to use a function numba does know: the
`math` module has `math.erf`, which numba compiles.

In [ ]:
@njit
def uses_math(x):
    return math.erf(x)


print("math.erf(0.5) in numba:", uses_math(0.5))
print("scipy.special.erf(0.5):", scipy.special.erf(0.5))

## 🎯 Summary

| Version | When to use it |
|---|---|
| 🐍 Python loop | never for heavy numerical work |
| 🧮 NumPy arrays | whenever the computation is a few whole-array operations |
| ⚡ numba `@njit` | loops that NumPy cannot express, such as time loops |
| ⚡⚡ numba `parallel=True` | independent steps with a lot of arithmetic each |

Remember: the first call compiles, a call with new types compiles again, and
`cache=True` keeps the machine code for the next session.

## 🧪 Try it yourself

1. In Section 3, change the grid to `d = 2**16` (keep `dx = 2 / d`, and make
   `dt` smaller, `dt = 1e-6`, so that the method stays stable). Does the
   ratio between NumPy and numba grow or shrink? Why?
2. Add `parallel=True` to `march_numba` and replace the inner `range(d)` by
   `prange(d)`. Why must the **outer** loop over time stay a `range`?
3. Restart the kernel and run Section 3 again. Is the first call of
   `march_numba` still slow? (Hint: `cache=True`.)